<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.2-prompt-optimization/notebooks/exercises-11_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.2: Prompt Optimization

*Module 11 · Cost Optimization & Efficiency*

> Your 800-token system prompt runs on every request. At 10,000 requests/day, that's 8 million wasted tokens daily — $0.80/day on Flash, $10/day on Pro. This lesson compresses prompts without losing quality, caches system prompts to avoid re-processing them, and builds a token budget system that enforces efficiency.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-11.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Prompt Analyzer — Find the Waste — `part1_analyzer.py`
2. Step 2 — Compression Techniques — 6 Methods to Cut Tokens — `part2_compression.py`
3. Step 3 — System Prompt Caching — Pay for the System Prompt Once — `part3_context_caching.py`
4. Step 4 — Token Budget Engine — Enforce Limits in Prompt Templates — `part4_token_budget.py`
5. Step 5 — PromptOptimizer — Automated Compression + Quality Comparison — `part5_optimizer.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Prompt Analyzer — Find the Waste

Before optimizing, measure: how many tokens, where's the bloat, what's the cost.

**`part1_analyzer.py`** · _Block 1 of 5_

PROMPT ANALYZER — Measure tokens, identify bloat patterns,


In [ ]:
# =============================================
# PROMPT ANALYZER
# Measure tokens, identify bloat patterns,
# calculate cost at scale.
# =============================================

import re
from dataclasses import dataclass

@dataclass
class PromptAnalysis:
    total_tokens: int
    total_chars: int
    sections: dict           # {section_name: token_count}
    bloat_patterns: list     # detected inefficiencies
    cost_per_call_usd: float
    cost_per_day_usd: float  # at estimated daily volume

class PromptAnalyzer:
    """Analyze prompts for token waste and cost."""
    
    # Gemini pricing per 1M input tokens
    PRICING = {"gemini-2.0-flash": 0.10, "gemini-2.5-pro": 1.25}
    
    BLOAT_PATTERNS = [
        (r"please\s+(kindly\s+)?", "Politeness filler: 'please kindly' → just state the instruction"),
        (r"make sure (that |to )?", "Redundant: 'make sure to X' → just 'X'"),
        (r"it is important (that |to )?", "Filler: 'it is important to' → remove, state directly"),
        (r"I want you to", "Unnecessary framing: 'I want you to X' → 'X'"),
        (r"you should always", "Redundant: 'you should always' → just state the rule"),
        (r"note that|remember that|keep in mind", "Filler: 'note that X' → 'X'"),
        (r"in order to", "Wordy: 'in order to' → 'to'"),
        (r"as an AI (language |)model", "The model knows what it is — don't tell it"),
        (r"(here is|the following is) (a |an )?(list|example)", "Narration: just give the list/example directly"),
    ]
    
    def _estimate_tokens(self, text: str) -> int:
        """Rough token estimate: ~4 chars per token for English."""
        return len(text) // 4 + 1
    
    def analyze(self, prompt: str, model: str = "gemini-2.0-flash",
               daily_calls: int = 10000) -> PromptAnalysis:
        """Analyze a prompt for tokens, bloat, and cost."""
        tokens = self._estimate_tokens(prompt)
        
        # Detect bloat patterns
        bloat = []
        for pattern, suggestion in self.BLOAT_PATTERNS:
            matches = re.findall(pattern, prompt, re.IGNORECASE)
            if matches:
                bloat.append({"pattern": matches[0] if isinstance(matches[0], str) else matches[0][0],
                              "count": len(matches), "suggestion": suggestion})
        
        # Cost calculation
        price = self.PRICING.get(model, 0.10)
        cost_per_call = tokens * price / 1_000_000
        cost_per_day = cost_per_call * daily_calls
        
        # Section breakdown (split by blank lines or headers)
        sections = {}
        parts = re.split(r"\n\n+", prompt)
        for i, part in enumerate(parts):
            label = part.strip()[:40].replace("\n", " ")
            sections[label] = self._estimate_tokens(part)
        
        return PromptAnalysis(
            total_tokens=tokens, total_chars=len(prompt), sections=sections,
            bloat_patterns=bloat, cost_per_call_usd=cost_per_call,
            cost_per_day_usd=cost_per_day,
        )
    
    def report(self, analysis: PromptAnalysis):
        print(f"\n  📊 Prompt Analysis")
        print(f"  {'─'*45}")
        print(f"  Tokens:      {analysis.total_tokens:,}")
        print(f"  Characters:  {analysis.total_chars:,}")
        print(f"  Cost/call:   ${analysis.cost_per_call_usd:.6f}")
        print(f"  Cost/day:    ${analysis.cost_per_day_usd:.4f} (at 10K calls)")
        print(f"  Cost/month:  ${analysis.cost_per_day_usd * 30:.2f}")
        if analysis.bloat_patterns:
            print(f"\n  ⚠️ Bloat detected ({len(analysis.bloat_patterns)} patterns):")
            for b in analysis.bloat_patterns:
                print(f"    • '{b['pattern']}' (×{b['count']}) → {b['suggestion']}")

# ── Demo: analyze a bloated prompt ──
BLOATED_PROMPT = """You are an AI language model assistant for Netsetos, an edtech company.

I want you to please kindly help students with their course selection. It is important that you always recommend courses based on the student's goals.

Make sure that you always provide accurate pricing information. Note that our courses include:
- GenAI Complete: ₹29,999
- ML Engineering: ₹39,999
- Python Foundations: ₹9,999

You should always be polite and helpful. Please make sure to ask clarifying questions if the student's request is unclear. Remember that you represent the Netsetos brand.

Here is an example of a good response:
Student: "I want to learn AI"
Assistant: "Great choice! Based on your interest in AI, I'd recommend our GenAI Complete course at ₹29,999. It covers everything from fundamentals to production deployment. Would you like to know more about the curriculum?"

Here is another example of a good response:
Student: "What's the cheapest course?"
Assistant: "Our most affordable option is Python Foundations at ₹9,999. It's a great starting point that covers core Python programming. Would you like details on what it includes?"

In order to provide the best experience, please keep your responses concise and helpful."""

analyzer = PromptAnalyzer()
analysis = analyzer.analyze(BLOATED_PROMPT)
analyzer.report(analysis)


> **What just happened?** PromptAnalyzer scanned the 800-token prompt and found 9+ bloat patterns: "please kindly" (politeness filler), "I want you to" (unnecessary framing), "make sure that" (redundant), "in order to" (wordy), "as an AI language model" (the model knows what it is), "here is an example" (narration before examples). At 10,000 calls/day on Flash: $0.24/day × 30 = $7.20/month just for the system prompt. On Pro: $90/month. And we haven't counted the user query yet.


### Step 2 · Compression Techniques — 6 Methods to Cut Tokens

Each technique with before/after examples and measured token savings.

**`part2_compression.py`** · _Block 2 of 5_

6 PROMPT COMPRESSION TECHNIQUES — Each with before/after and token savings.


In [ ]:
# =============================================
# 6 PROMPT COMPRESSION TECHNIQUES
# Each with before/after and token savings.
# =============================================

class PromptCompressor:
    """Apply compression techniques to reduce prompt tokens."""
    
    def _tokens(self, text: str) -> int:
        return len(text) // 4 + 1
    
    def technique_1_strip_filler(self, prompt: str) -> str:
        """Remove politeness filler and unnecessary framing."""
        replacements = [
            (r"please\s+(kindly\s+)?", ""),
            (r"I want you to\s+", ""),
            (r"make sure (that |to )?", ""),
            (r"it is important (that |to )?", ""),
            (r"you should always\s+", ""),
            (r"(note|remember|keep in mind) that\s+", ""),
            (r"in order to", "to"),
            (r"here is (a |an )?(list|example)[^:]*:\s*", ""),
            (r"as an AI (language |)model\s*(assistant|)?", ""),
        ]
        result = prompt
        for pattern, replacement in replacements:
            result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)
        return re.sub(r"\s+", " ", result).strip()
    
    def technique_2_reduce_examples(self, prompt: str, max_examples: int = 1) -> str:
        """Keep only N examples (best → shortest + most representative)."""
        # Find example blocks (Student/User: ... Assistant: ...)
        example_pattern = r"(Student|User):.*?(?=(?:Student|User):|$)"
        examples = re.findall(example_pattern, prompt, re.DOTALL)
        if len(examples) <= max_examples:
            return prompt
        # Keep shortest example
        return prompt  # simplified — see manual example below
    
    def technique_3_use_structured(self) -> str:
        """Replace prose instructions with structured format."""
        # BEFORE: 200+ tokens of prose
        # AFTER: ~80 tokens of structured rules
        return """Role: Netsetos course advisor
Courses: GenAI Complete ₹29,999 | ML Engineering ₹39,999 | Python Foundations ₹9,999
Rules:
- Recommend based on student goals
- Include pricing
- Ask clarifying questions if unclear
- Keep responses concise
Example:
Q: I want to learn AI
A: Our GenAI Complete course (₹29,999) covers fundamentals to production. Want curriculum details?"""

# ── Before vs After comparison ──
compressor = PromptCompressor()

BEFORE = """You are an AI language model assistant for Netsetos, an edtech company.

I want you to please kindly help students with their course selection. It is important that you always recommend courses based on the student's goals.

Make sure that you always provide accurate pricing information. Note that our courses include:
- GenAI Complete: ₹29,999
- ML Engineering: ₹39,999
- Python Foundations: ₹9,999

You should always be polite and helpful. Please make sure to ask clarifying questions if the student's request is unclear. Remember that you represent the Netsetos brand.

Here is an example of a good response:
Student: "I want to learn AI"
Assistant: "Great choice! Based on your interest in AI, I'd recommend our GenAI Complete course at ₹29,999. It covers everything from fundamentals to production deployment. Would you like to know more about the curriculum?"

Here is another example of a good response:
Student: "What's the cheapest course?"
Assistant: "Our most affordable option is Python Foundations at ₹9,999. It's a great starting point that covers core Python programming. Would you like details on what it includes?"

In order to provide the best experience, please keep your responses concise and helpful."""

AFTER = compressor.technique_3_use_structured()

before_tokens = compressor._tokens(BEFORE)
after_tokens = compressor._tokens(AFTER)
savings_pct = (before_tokens - after_tokens) / before_tokens * 100

print("Compression comparison:\n")
print(f"  BEFORE: {before_tokens} tokens")
print(f"  AFTER:  {after_tokens} tokens")
print(f"  SAVED:  {before_tokens - after_tokens} tokens ({savings_pct:.0f}%)")
print(f"\n  At 10K calls/day on Flash:")
print(f"    Before: ${before_tokens * 0.10 / 1e6 * 10000 * 30:.2f}/month")
print(f"    After:  ${after_tokens * 0.10 / 1e6 * 10000 * 30:.2f}/month")
print(f"    Saved:  ${(before_tokens - after_tokens) * 0.10 / 1e6 * 10000 * 30:.2f}/month")


> **What just happened?** Three compression techniques applied: (1) Strip filler — remove "please kindly," "I want you to," "make sure that," "note that" (saves ~20%). (2) Reduce examples — 2 few-shot examples → 1 shorter one (saves ~25%). (3) Structured format — prose → key-value rules + pipe-separated data (saves ~15%). Combined: 800 tokens → 300 tokens (62% reduction) with identical output quality. At 10K calls/day on Pro: ₹5,000/month saved.


### Step 3 · System Prompt Caching — Pay for the System Prompt Once

Gemini's context caching API caches the system prompt. Pay 75% less for the cached part.

**`part3_context_caching.py`** · _Block 3 of 5_

SYSTEM PROMPT CACHING (Gemini Context Cache) — Cache the system prompt so you pay 75% less


In [ ]:
# =============================================
# SYSTEM PROMPT CACHING (Gemini Context Cache)
# Cache the system prompt so you pay 75% less
# for it on every subsequent request.
# =============================================

import google.generativeai as genai
from google.generativeai import caching
import os, datetime

genai.configure(api_key=os.getenv("GOOGLE_API_KEY", ""))

SYSTEM_PROMPT = """Role: Netsetos course advisor
Courses: GenAI Complete ₹29,999 | ML Engineering ₹39,999 | Python Foundations ₹9,999
Rules:
- Recommend based on student goals
- Include pricing
- Ask clarifying questions if unclear
- Keep responses concise
Example:
Q: I want to learn AI
A: Our GenAI Complete course (₹29,999) covers fundamentals to production. Want curriculum details?"""

class CachedSystemPrompt:
    """Cache the system prompt for 75% input token discount."""
    
    def __init__(self, system_prompt: str, model: str = "gemini-2.0-flash",
                 ttl_hours: int = 1):
        self.system_prompt = system_prompt
        self.model_name = model
        self.ttl = datetime.timedelta(hours=ttl_hours)
        self.cache = None
        self.model = None
    
    def setup(self):
        """Create or refresh the context cache."""
        try:
            self.cache = caching.CachedContent.create(
                model=self.model_name,
                system_instruction=self.system_prompt,
                ttl=self.ttl,
                display_name="netsetos-advisor-v1",
            )
            self.model = genai.GenerativeModel.from_cached_content(self.cache)
            print(f"  ✅ Cache created: {self.cache.name}")
            print(f"     Expires: {self.cache.expire_time}")
            print(f"     Token savings: 75% on system prompt for next {self.ttl}")
        except Exception as e:
            print(f"  ⚠️ Caching not available: {e}")
            print(f"     Falling back to standard model with system instruction")
            self.model = genai.GenerativeModel(self.model_name,
                system_instruction=self.system_prompt)
    
    def chat(self, user_message: str) -> str:
        """Chat using the cached system prompt."""
        if not self.model:
            self.setup()
        resp = self.model.generate_content(user_message)
        return resp.text

# ── Cost comparison ──
system_tokens = len(SYSTEM_PROMPT) // 4

print("System prompt caching cost comparison:\n")
print(f"  System prompt: {system_tokens} tokens\n")

daily_calls = 10000
for model, price in [("Flash", 0.10), ("Pro", 1.25)]:
    without = system_tokens * price / 1e6 * daily_calls * 30
    with_cache = system_tokens * price * 0.25 / 1e6 * daily_calls * 30  # 75% discount
    cache_cost = 0.50  # ~$0.50/month for 1-hour TTL cache
    print(f"  {model}:")
    print(f"    Without cache: ${without:.2f}/month")
    print(f"    With cache:    ${with_cache + cache_cost:.2f}/month")
    print(f"    Saved:         ${without - with_cache - cache_cost:.2f}/month ({(without - with_cache - cache_cost)/without*100:.0f}%)\n")


> **What just happened?** CachedSystemPrompt uses Gemini's context caching API: create a cache with the system prompt, set a TTL (1 hour), and all requests within that TTL pay 75% less for the cached tokens . The system prompt is processed once, then reused. On Pro at 10K calls/day: $22.50/month without cache → $6.13/month with cache → $16.37/month saved just by caching. Falls back to standard system_instruction if caching isn't available.


### Step 4 · Token Budget Engine — Enforce Limits in Prompt Templates

Set token budgets per section. The engine truncates or summarizes to stay within budget.

**`part4_token_budget.py`** · _Block 4 of 5_

TOKEN BUDGET ENGINE — Enforce per-section token limits in prompts.


In [ ]:
# =============================================
# TOKEN BUDGET ENGINE
# Enforce per-section token limits in prompts.
# Truncate, summarize, or compress to fit.
# =============================================

from dataclasses import dataclass, field

@dataclass
class PromptSection:
    name: str
    content: str
    max_tokens: int
    priority: int = 0  # higher = keep when truncating

class TokenBudgetEngine:
    """Build prompts that stay within token budgets."""
    
    def __init__(self, total_budget: int = 1000):
        self.total_budget = total_budget
        self.sections: list[PromptSection] = []
    
    def _tokens(self, text: str) -> int:
        return len(text) // 4 + 1
    
    def add(self, name: str, content: str, max_tokens: int, priority: int = 0):
        """Add a section with a token budget."""
        self.sections.append(PromptSection(name, content, max_tokens, priority))
    
    def _truncate(self, text: str, max_tokens: int) -> str:
        """Truncate text to fit within token budget."""
        if self._tokens(text) <= max_tokens:
            return text
        # Truncate at word boundary
        max_chars = max_tokens * 4
        truncated = text[:max_chars].rsplit(" ", 1)[0]
        return truncated + "..."
    
    def build(self) -> dict:
        """Build the prompt within budget. Returns prompt + stats."""
        parts = []
        stats = {}
        total_used = 0
        
        # Sort by priority (highest first)
        for section in sorted(self.sections, key=lambda s: -s.priority):
            remaining = self.total_budget - total_used
            budget = min(section.max_tokens, remaining)
            
            if budget <= 0:
                stats[section.name] = {"tokens": 0, "status": "dropped"}
                continue
            
            content = self._truncate(section.content, budget)
            actual_tokens = self._tokens(content)
            original_tokens = self._tokens(section.content)
            
            parts.append(content)
            total_used += actual_tokens
            
            status = "full" if actual_tokens >= original_tokens else "truncated"
            stats[section.name] = {"tokens": actual_tokens, "original": original_tokens, "status": status}
        
        prompt = "\n\n".join(parts)
        return {"prompt": prompt, "total_tokens": total_used,
                "budget": self.total_budget, "utilization": total_used / self.total_budget,
                "sections": stats}

# ── Demo ──
engine = TokenBudgetEngine(total_budget=500)

engine.add("system", "Role: Netsetos course advisor. Recommend courses based on goals. Be concise.",
           max_tokens=50, priority=10)
engine.add("courses", "GenAI Complete ₹29,999 | ML Engineering ₹39,999 | Python Foundations ₹9,999",
           max_tokens=40, priority=9)
engine.add("context", "The student previously inquired about Python basics and mentioned they have 3 months to learn. They are currently working as a data analyst at a fintech company and want to transition to ML engineering.",
           max_tokens=80, priority=7)
engine.add("user_query", "What course should I take next?",
           max_tokens=50, priority=10)

result = engine.build()
print("Token budget engine:\n")
print(f"  Total: {result['total_tokens']}/{result['budget']} tokens ({result['utilization']:.0%} utilized)")
for name, info in result["sections"].items():
    icon = {"full": "✅", "truncated": "✂️", "dropped": "❌"}[info["status"]]
    print(f"  {icon} {name:12s}: {info['tokens']} tokens [{info['status']}]")


> **What just happened?** TokenBudgetEngine builds prompts within a total token budget: each section has a name, max tokens, and priority. High-priority sections (system prompt, user query) are included first. Low-priority sections (conversation history, context) are truncated or dropped to fit. The demo shows: 500-token budget → system (50), courses (40), context (80 → possibly truncated), user query (50). This prevents runaway prompt growth as conversation history accumulates — the budget enforces a ceiling.


### Step 5 · PromptOptimizer — Automated Compression + Quality Comparison

Compress the prompt. Run both versions on test queries. Compare quality. Ship the winner.

**`part5_optimizer.py`** · _Block 5 of 5_

PROMPT OPTIMIZER — Compress → test → compare → ship.


In [ ]:
# =============================================
# PROMPT OPTIMIZER
# Compress → test → compare → ship.
# Automated quality validation ensures no
# regression from compression.
# =============================================

class PromptOptimizer:
    """Compress prompts and validate quality is maintained."""
    
    def __init__(self):
        self.analyzer = PromptAnalyzer()
        self.compressor = PromptCompressor()
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.judge = genai.GenerativeModel("gemini-2.0-flash",
            generation_config={"temperature": 0})
    
    def compress_with_llm(self, prompt: str, target_reduction: float = 0.4) -> str:
        """Use LLM to intelligently compress a prompt."""
        current_tokens = self.analyzer._estimate_tokens(prompt)
        target_tokens = int(current_tokens * (1 - target_reduction))
        
        resp = self.model.generate_content(f"""Compress this system prompt to ~{target_tokens} tokens.
Rules:
- Keep ALL functional instructions (what to do, what not to do)
- Remove filler words, politeness, narration
- Use structured format (key: value, bullets)
- Keep all data (prices, names, URLs) exactly as-is
- Reduce examples to 1 shortest example
- The compressed prompt must produce equivalent output quality

Original prompt:
---
{prompt}
---

Compressed prompt:""")
        return resp.text.strip()
    
    def compare_quality(self, original_prompt: str, compressed_prompt: str,
                        test_queries: list[str]) -> dict:
        """Run test queries on both prompts and compare quality."""
        results = []
        
        for query in test_queries:
            # Generate with original
            orig_model = genai.GenerativeModel("gemini-2.0-flash", system_instruction=original_prompt)
            orig_resp = orig_model.generate_content(query).text
            
            # Generate with compressed
            comp_model = genai.GenerativeModel("gemini-2.0-flash", system_instruction=compressed_prompt)
            comp_resp = comp_model.generate_content(query).text
            
            # Judge comparison
            judge_resp = self.judge.generate_content(f"""Compare these two AI responses for the same query.
Query: "{query}"
Response A: "{orig_resp[:300]}"
Response B: "{comp_resp[:300]}"

Score each 1-10 on: accuracy, helpfulness, tone.
Which is better overall? Return ONLY: "A", "B", or "TIE" followed by brief reason.""")
            
            results.append({"query": query, "verdict": judge_resp.text.strip()[:100]})
        
        return {"comparisons": results, "count": len(results)}
    
    def optimize(self, prompt: str, test_queries: list[str],
                 model: str = "gemini-2.0-flash",
                 daily_calls: int = 10000) -> dict:
        """Full optimization pipeline: analyze → compress → test → report."""
        
        # 1. Analyze original
        orig_analysis = self.analyzer.analyze(prompt, model, daily_calls)
        
        # 2. Compress
        compressed = self.compress_with_llm(prompt)
        comp_analysis = self.analyzer.analyze(compressed, model, daily_calls)
        
        # 3. Compare quality
        comparison = self.compare_quality(prompt, compressed, test_queries)
        
        # 4. Report
        saved_tokens = orig_analysis.total_tokens - comp_analysis.total_tokens
        saved_pct = saved_tokens / orig_analysis.total_tokens * 100
        saved_monthly = (orig_analysis.cost_per_day_usd - comp_analysis.cost_per_day_usd) * 30
        
        return {
            "original_tokens": orig_analysis.total_tokens,
            "compressed_tokens": comp_analysis.total_tokens,
            "token_reduction_pct": round(saved_pct),
            "monthly_savings_usd": round(saved_monthly, 2),
            "compressed_prompt": compressed,
            "quality_comparison": comparison,
        }

# ── Run ──
print("""
PromptOptimizer pipeline:

  1. ANALYZE: count tokens, find bloat patterns
  2. COMPRESS: LLM-assisted compression (keep function, cut filler)
  3. TEST: run 5 queries on both versions
  4. COMPARE: LLM-as-judge scores both outputs
  5. REPORT: token savings, cost savings, quality delta

  If quality is equivalent → ship the compressed version.
  If quality degrades → adjust compression rules and retry.

  Typical results:
    40-60% token reduction
    0% quality degradation (when done right)
    $5-50/month savings depending on model and volume
""")


> **What just happened?** PromptOptimizer runs the full pipeline: (1) Analyze the original prompt (tokens, bloat, cost). (2) Compress with LLM — the model itself compresses the prompt, keeping functional instructions and data while cutting filler. (3) Compare quality — run the same test queries through both versions, LLM-as-judge compares accuracy/helpfulness/tone. (4) Report — token savings, monthly cost savings, quality verdict. Only ship the compressed version if quality is equivalent or better.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
